In [2]:
import time
import gpytoolbox as gpy
import numpy as np
from collections import defaultdict
import math
import h5py
from tqdm import tqdm
# Imports
import os
import sys
import gc

In [3]:
sdf_numpy = np.load('37361.npy')

In [32]:
def mesh_grid(grid_size: int, normalize=False):
    """create mesh grid with default indexing"""
    xx, yy, zz = np.mgrid[:grid_size, :grid_size, :grid_size]
    grid_3d = np.column_stack((xx.flatten(), yy.flatten(), zz.flatten()))
    if normalize:
        return 2 * (grid_3d / (grid_size - 1)) - 1
    return grid_3d

U = mesh_grid(33, True)
U_int = mesh_grid(33)
S = 2*sdf_numpy[U_int[:, 0], U_int[:, 1], U_int[:, 2]]

In [5]:
res=32
j=res
gx, gy, gz = np.meshgrid(
                np.linspace(-1, 1, j+1), np.linspace(-1, 1, j+1), np.linspace(-1, 1, j+1))
U = np.vstack((gx.flatten(), gy.flatten(), gz.flatten())).T
U_int = (U*(res/2) + (res/2)).astype(np.int32)

In [25]:
S = 2*sdf_numpy[U_int[:, 0], U_int[:, 1], U_int[:, 2]]

In [33]:
Vr, Fr = gpy.reach_for_the_arcs( U, S, verbose=True, parallel=True, return_point_cloud=False, max_points_per_sphere=3, fine_tune_iters=3)
            

 --- Starting Reach for the Arcs --- 
SDF to point cloud...
  positive spheres
  Rasterization...
    Rasterizing on GPU.
  Rasterization took 1.0454363822937012s

  Locally make feasible...
    34528 / 35436 points are feasible.
  Locally make feasible took 1.313546895980835s

  negative spheres
  Rasterization...
    Rasterizing on CPU.
  Rasterization took 0.11045384407043457s

  Locally make feasible...
    501 / 501 points are feasible.
  Locally make feasible took 0.008245468139648438s

SDF to point cloud took 2.482623815536499s

Fine tuning point cloud...
  Fine tune called with 35029 / 35937 feasible points.
  After fine tuning iter 0, we have 37447 points.
  After fine tuning iter 1, we have 38323 points.
  After fine tuning iter 2, we have 38566 points.
Fine tuning point cloud took 22.582343339920044s

Converting point cloud to mesh...
Converting point cloud to mesh took 4.7571282386779785s

 --- Finished Reach for the Arcs --- 
Total elapsed time: 29.822753190994263s


In [34]:
def export_obj(nv: np.ndarray, nf: np.ndarray, name: str, export_lines=False):
    try:
        file = open(name, "x")
    except:
        file = open(name, "w")
    for e in nv:
        file.write("v {} {} {}\n".format(*e))
    file.write("\n")
    for face in nf:
        header = "l " if export_lines else "f "
        file.write(header + " ".join([str(fi + 1) for fi in face]) + "\n")
    file.write("\n")

In [35]:
export_obj(Vr, Fr, '37361_rfta_2.obj')